# 🧠 Build a Mini-LLM (Baby GPT) from Scratch in PyTorch

<a href="https://colab.research.google.com/github/Ali747711/CV/blob/master/mini_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Welcome! In this notebook, we will build a **real, working Large Language Model (Decoder-Only Transformer)** from the ground up using **pure PyTorch**. 

This is the exact same fundamental architecture that powers **GPT-2, GPT-3, GPT-4, and LLaMA**, scaled down so you can train it directly on your laptop in about **2–3 minutes**.

---

### 📖 Key Terms & Definitions

Before we write code, here are the essential concepts you need to know:

1. **LLM (Large Language Model)**: A neural network trained on vast amounts of text to predict the *next token* given a sequence of previous tokens (autoregressive modeling).
2. **Token & Tokenizer**: A "token" is the basic unit of text an AI reads. A tokenizer converts human text into integers (numbers) that computers can calculate with, and turns those numbers back into human words.
3. **Context Length (`block_size`)**: The maximum number of consecutive tokens the model can look at simultaneously when predicting the next token.
4. **Embedding**: Converting each token ID into a dense vector of numbers (e.g., 64 numbers) that represent semantic meaning.
5. **Self-Attention**: The core mechanism of modern AI. It allows tokens in a sentence to look at other tokens and decide which ones are important (e.g., in "The bank of the river", "bank" attends to "river" to know it's not a financial bank).
6. **Causal Masking**: A rule that prevents tokens from looking into the *future*. When predicting token #5, the model is only allowed to look at tokens #1, #2, #3, and #4.
7. **Transformer Block**: The repeating building block of GPT, containing **Multi-Head Self-Attention** + **Feed-Forward Layers** + **Residual Connections** + **LayerNorm**.

## 0. Environment Setup & Device Configuration

We detect whether your machine has an Apple Silicon GPU (`mps`), an Nvidia GPU (`cuda`), or fallback to `cpu`.

In [1]:
# Optional: uncomment if running in an environment without torch installed:
# !pip install torch matplotlib

import torch
import torch.nn as nn
from torch.nn import functional as F
import urllib.request
import os

# Set random seed for reproducibility
torch.manual_seed(1337)

# Device-agnostic setup (Supports Apple Silicon Mac GPU 'mps'!)
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"🚀 Using compute device: {device}")

🚀 Using compute device: mps


---
## 1. Dataset & Tokenizer Definition

### 📚 What is a Tokenizer?
Computers cannot process raw strings like `"Hello"`. A **tokenizer** maps text into numbers:
* In production LLMs (like GPT-4), tokenizers use **Byte-Pair Encoding (BPE)** where subwords like `"un"`, `"break"`, `"able"` each get an integer ID (~50,000 unique tokens).
* In our Mini-LLM, we build a **Character-Level Tokenizer**: every unique character (`'a'`, `'b'`, `' '`, `'\n'`) gets its own integer ID. 

Our dataset is **Tiny Shakespeare** (excerpts from Shakespeare's plays). It contains **65 unique characters**.

In [2]:
# 1. Download Tiny Shakespeare dataset if not already present
dataset_path = 'input.txt'
if not os.path.exists(dataset_path):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, dataset_path)
    print("Downloaded input.txt successfully!")

# 2. Read the text
with open(dataset_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters in dataset: {len(text):,}")

# 3. Extract unique characters (our vocabulary)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"Unique characters: {''.join(chars)}")

# 4. Create mapping from character -> integer (stoi) and integer -> character (itos)
stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }

# Encoder: string to list of integers
encode = lambda s: [stoi[c] for c in s]
# Decoder: list of integers to string
decode = lambda l: ''.join([itos[i] for i in l])

# Test our tokenizer:
sample_text = "Hello world!"
encoded = encode(sample_text)
decoded = decode(encoded)
print(f"\nOriginal : '{sample_text}'")
print(f"Encoded  : {encoded}")
print(f"Decoded  : '{decoded}'")

Total characters in dataset: 1,115,394
Vocabulary size: 65
Unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Original : 'Hello world!'
Encoded  : [20, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
Decoded  : 'Hello world!'


---
## 2. Train/Validation Split & How LLMs Learn Sequences

### 📚 How do LLMs Predict the Next Token?
An LLM is trained on sequences of tokens of length `block_size` (context length). 

For example, if `block_size = 8` and the text is `"hello wo"`:
* Input $X$: `[h, e, l, l, o,  , w]`
* Target $Y$: `[e, l, l, o,  , w, o]` *(shifted right by 1!)*

This means in **one single sequence of 8 tokens**, the model actually learns **8 separate predictions simultaneously**:
* Given `"h"`, predict `"e"`
* Given `"he"`, predict `"l"`
* Given `"hel"`, predict `"l"`
* ...
* Given `"hello w"`, predict `"o"`

In [3]:
# Convert entire text into a PyTorch LongTensor
data = torch.tensor(encode(text), dtype=torch.long)

# 90% train, 10% validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Train tokens: {len(train_data):,} | Validation tokens: {len(val_data):,}")

# Hyperparameters for batching
batch_size = 32 # How many independent sequences will we process in parallel?
block_size = 64 # What is the maximum context length for predictions?

def get_batch(split):
    # Generate a small batch of data of inputs x and targets y
    dataset = train_data if split == 'train' else val_data
    # Pick random starting indices in the dataset
    ix = torch.randint(len(dataset) - block_size, (batch_size,))
    x = torch.stack([dataset[i:i+block_size] for i in ix])
    y = torch.stack([dataset[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# Inspect a single sample batch
xb, yb = get_batch('train')
print(f"Inputs shape (batch_size, block_size): {xb.shape}")
print(f"Targets shape (batch_size, block_size): {yb.shape}")

print("\nSample sequence decoded:")
print("Input (x):", decode(xb[0][:20].tolist()))
print("Target(y):", decode(yb[0][:20].tolist()))

Train tokens: 1,003,854 | Validation tokens: 111,540
Inputs shape (batch_size, block_size): torch.Size([32, 64])
Targets shape (batch_size, block_size): torch.Size([32, 64])

Sample sequence decoded:
Input (x): le. Think with thyse
Target(y): e. Think with thysel


---
## 3. The Core of an LLM: Self-Attention Explained

### 📚 What is Self-Attention?
In a sentence, words need to share information with each other to understand context.
For each token, Self-Attention computes three vectors:
1. **Query ($Q$)**: *"What kind of information am I looking for?"*
2. **Key ($K$)**: *"What kind of information do I contain?"*
3. **Value ($V$)**: *"If you care about me, here is the information I will give you."*

#### The Attention Formula:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

1. We multiply $Q$ by $K^T$ to compute an **affinity / attention score** between every pair of tokens.
2. We divide by $\sqrt{d_k}$ (scaling factor) so variance doesn't explode.
3. **Causal Mask (`tril`)**: Because an LLM can only predict the *future* from the *past*, we mask all future positions with $-\infty$ before applying `softmax`. The softmax of $-\infty$ is `0`, guaranteeing future tokens cannot leak information backwards!
4. We multiply the softmax probabilities by $V$ to get the final contextualized representation.

In [4]:
class Head(nn.Module):
    """ One single head of Self-Attention """

    def __init__(self, head_size, n_embd, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        # Lower-triangular matrix for causal masking (registered as buffer, not parameter)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Input shape: (Batch, Time/Sequence, Channels/n_embd)
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        
        # Compute attention scores ("affinities"): (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        
        # Causal mask: block out the future by replacing upper triangle with -inf
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        
        # Softmax turns raw scores into probabilities that sum to 1
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        
        # Perform weighted aggregation of values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, T) @ (B, T, head_size) -> (B, T, head_size)
        return out

---
## 4. Multi-Head Attention, Feed-Forward, & Residual Connections

### 📚 Why Multi-Head Attention?
A single attention head can only focus on one kind of relationship at a time (e.g. finding who is speaking). 
By creating **multiple attention heads in parallel**, different heads can simultaneously look for:
* Grammatical syntax (subject-verb agreement)
* Rhyme or meter in poetry
* Dialogue character names
* Punctuation patterns

We then concatenate their outputs and project back using a linear layer.

### 📚 What are Residual Connections & LayerNorm?
* **Residual Connections ($x + \text{layer}(x)$)**: By adding the original input back to the processed output, gradients can flow directly backwards during backpropagation without diminishing, allowing models to be stacked deep.
* **Feed-Forward Network (FFN)**: A simple 2-layer multi-layer perceptron (MLP) applied to each token independently so it can "think" about the information gathered by attention.
* **LayerNorm**: Normalizes values across the channel dimension so activations don't explode or vanish.

In [5]:
class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention running in parallel """

    def __init__(self, num_heads, head_size, n_embd, block_size, dropout=0.1):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Concatenate outputs of all heads along channel dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # Linear projection back to embedding dimension
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """ A simple linear layer followed by non-linearity """

    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        # In the original Transformer paper, inner dimension is expanded 4x
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ One Transformer Block: Communication (Attention) followed by Computation (FFN) """

    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embd, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Pre-LN architecture with residual connections
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

---
## 5. The Complete MiniGPT Architecture

### 📚 Putting All the Pieces Together
The complete Decoder-Only Transformer consists of:
1. **Token Embedding Table**: Maps token ID $i \to$ vector of size `n_embd`.
2. **Position Embedding Table**: Maps sequence index $t \in [0, \text{block\_size}-1] \to$ vector of size `n_embd`.
3. **Sequential Transformer Blocks**: Stacking multiple `Block` layers.
4. **Final LayerNorm & Language Modeling Head**: Projects the final hidden states back to vocabulary size (`vocab_size` = 65) to produce **logits** (unnormalized log probabilities for each possible next character).
5. **Autoregressive Generation (`generate`)**: Feeds existing tokens in, gets next token logits, applies softmax to sample the next character, appends it, and repeats!

In [6]:
class MiniGPT(nn.Module):

    def __init__(self, vocab_size, n_embd=64, n_head=4, n_layer=4, block_size=64, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        
        # 1. Token and Position Embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        
        # 2. Stack of Transformer Blocks
        self.blocks = nn.Sequential(*[
            Block(n_embd=n_embd, n_head=n_head, block_size=block_size, dropout=dropout)
            for _ in range(n_layer)
        ])
        
        # 3. Final LayerNorm & Linear projection to vocab
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Token embeddings + Positional embeddings
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)
        
        # Pass through Transformer blocks
        x = self.blocks(x) # (B, T, n_embd)
        x = self.ln_f(x)   # (B, T, n_embd)
        
        # Project to vocabulary logits
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            # Reshape for cross_entropy: (Batch * Time, Vocab_Size)
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=1.0):
        # idx is (B, T) array of indices in current context
        for _ in range(max_new_tokens):
            # Crop idx to the last block_size tokens so positional embeddings don't overflow
            idx_cond = idx[:, -self.block_size:]
            
            # Forward pass to get logits
            logits, _ = self(idx_cond)
            
            # Focus only on the last time step (the predicted next token)
            logits = logits[:, -1, :] / temperature # (B, C)
            
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            
            # Append sampled index to running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

---
## 6. Testing the Untrained Model (Before Training)

Before training, the model's weights are initialized randomly. Let's inspect the model parameter count and generate text to see what pure random guessing looks like.

In [7]:
# Hyperparameters
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.1

model = MiniGPT(
    vocab_size=vocab_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    block_size=block_size,
    dropout=dropout
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params:,}")

# Generate from untrained model (starting with newline character token)
print("\n--- GENERATED TEXT BEFORE TRAINING (Random Guessing) ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device) # token index 0
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))
print("---------------------------------------------------------")

Total model parameters: 211,777

--- GENERATED TEXT BEFORE TRAINING (Random Guessing) ---



wfB.nQ?,xo:EvUMrkctHzEgZk ewcNONzJQmIFZp'.FfR;LJvw,VREwoJy
DubZAtw&a!cfxdZnbNsKNsWXsM.O!FM?SqZnQ? tt
wbcp:wyvSsYIyBTeXEXOOYfRj'IafFaNHVFaSlsqx!BnqTpK,z:SxtJTtxs'i;DHOz.XqTNMbqlFnwxTuH,UWv&WGXRSZB,GwMf
---------------------------------------------------------


---
## 7. Training the Model (Next-Token Prediction)

### 📚 Training Mechanism
* **Loss Function**: `CrossEntropyLoss` measures how far the model's predicted probability distribution is from the actual next character.
* **Optimizer**: `AdamW` (Adaptive Moment Estimation with Decoupled Weight Decay), the standard optimizer for LLMs.
* We evaluate every `eval_interval` steps on both train and validation sets without calculating gradients (`torch.no_grad()`).

In [8]:
# Training configuration
max_iters = 1500       # Fast training (~1-2 mins on Apple Silicon / CPU)
eval_interval = 300
eval_iters = 100
learning_rate = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print("Starting training...")
for iter in range(max_iters):
    # Periodically evaluate loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {iter:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Forward pass & loss
    logits, loss = model(xb, yb)
    
    # Backward pass & optimizer step
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("✅ Training complete!")

Starting training...


Step    0 | Train Loss: 4.3166 | Val Loss: 4.3219


Step  300 | Train Loss: 2.4096 | Val Loss: 2.4161


Step  600 | Train Loss: 2.2012 | Val Loss: 2.2203


Step  900 | Train Loss: 2.0563 | Val Loss: 2.1016


Step 1200 | Train Loss: 1.9553 | Val Loss: 2.0182


Step 1499 | Train Loss: 1.8870 | Val Loss: 1.9768
✅ Training complete!


---
## 8. Generating Shakespeare Text (After Training)

Now that the weights have been optimized, let's generate text! Notice how the model has learned:
1. English words and spaces.
2. Shakespearean punctuation and poetic cadence.
3. Character dialogue formatting (e.g., `GLOUCESTER:`, `KING HENRY VI:`).

In [9]:
model.eval()

# 1. Unconditional generation (starting from newline)
print("=== GENERATED SAMPLE 1 (Unconditional) ===")
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text = decode(model.generate(context, max_new_tokens=400, temperature=0.8)[0].tolist())
print(generated_text)
print("\n" + "="*50 + "\n")

# 2. Prompt-conditioned generation (Giving the model a starting prompt!)
prompt = "ROMEO:\n"
prompt_encoded = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

print(f"=== GENERATED SAMPLE 2 (Prompt: '{prompt.strip()}') ===")
generated_prompt = decode(model.generate(prompt_encoded, max_new_tokens=400, temperature=0.8)[0].tolist())
print(generated_prompt)

=== GENERATED SAMPLE 1 (Unconditional) ===




Should the this pamond crise.
 stencong he morrom light: my theare theake by arame.
Whrill is what that quortaly prised bets.

MARD:
Hom and shall the kno! chat I my beade,
As I nett that whit be do the is from chingen:
So word, not then I lay ut not wheere as Mones
Untsellay, the will and for it upletage.

POPERCELA:
You swell beforrof I all be that thane the well:
Well as nast he cailcens your 


=== GENERATED SAMPLE 2 (Prompt: 'ROMEO:') ===


ROMEO:
And then as he ball hush the preconter the,
And I lough mader the thou deat your not hear.

CORCIINGEY:
Bet to him be love ried, cand stime their earther
A dable of stand the your try ballingeon.

SADWARD ELLY:
Nukere, lick it is the with mulser, say minding!

CARDIIO:
Frow ciuse? Saseth uf betion, that be one the lare
theark
And wand my laved lod forther is betille.

LORGBRENCE:
And it the fill t


---
## 9. Key Takeaways & Comparison to Commercial LLMs

Congratulations! 🎉 You just built and trained a genuine **Decoder-Only Transformer LLM** from scratch.

### How does this compare to GPT-4, Claude, or LLaMA?

| Feature | Our MiniGPT | Modern Commercial LLMs |
| :--- | :--- | :--- |
| **Tokenizer** | Characters (65 tokens) | Byte-Pair Encoding / Tiktoken (~50,000–100,000 tokens) |
| **Layers** | 4 Blocks | 32 to 128+ Blocks |
| **Embedding Size** | 64 | 4,096 to 12,288+ |
| **Context Window**| 64 characters | 8,000 to 2,000,000 tokens |
| **Training Steps**| 1. Pretraining (what we did here) | 1. Pretraining (Next-token prediction)<br>2. SFT (Instruction Fine-Tuning)<br>3. RLHF (Alignment for safety & helpfulness) |

The core mathematical algorithm you wrote here — **Self-Attention ($Q, K, V$)**, **LayerNorm**, **Residual Connections**, and **Next-Token Cross-Entropy** — is the exact same foundation powering every frontier AI today!

In [10]:
# Save your trained model weights!
torch.save(model.state_dict(), "mini_gpt_shakespeare.pth")
print("Saved model checkpoint to 'mini_gpt_shakespeare.pth'")

Saved model checkpoint to 'mini_gpt_shakespeare.pth'
